In [1]:

import fitz  # PyMuPDF (ATENÇÃO: pacote correto é PyMuPDF; import é 'fitz')
import re
import os
import unicodedata
from datetime import datetime
from openpyxl import Workbook

# =========================
# CONFIGURAÇÕES
# =========================
pasta_pdf = r"C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\ABERTURA DE REQUISIÇÃO\LOJAS NOVAS\LUMICENTER\pdfs_auto_lumi"

# Data atual no formato ddMMyyyy para a coluna "DATA"
data_hoje = datetime.today().strftime("%d%m%Y")

# =========================
# UTILITÁRIAS
# =========================
def clean_number(num_str: str):
    """
    Converte números em formato BR para float.
      - '1.234,56' -> 1234.56
      - '488,77'   -> 488.77
      - '123'      -> 123.0
    Retorna None se não converter.
    """
    if not num_str:
        return None
    s = str(num_str).strip()
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except Exception:
        return None

def normaliza(s: str) -> str:
    """
    Remove acentos e coloca em minúsculas para facilitar matching.
    """
    if not s:
        return ""
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return s.lower()

# =========================
# PADRÕES DE MATERIAL (regex -> código)
# =========================
# Observação:
# - quadrad[oa]s? casa 'quadrado', 'quadrada', 'quadrados', 'quadradas'
# - pret[oa]s?   casa 'preto', 'preta', 'pretos', 'pretas'
material_patterns = [
    (re.compile(r"\bquadrad[oa]s?\b"), "32434"),  # luminária quadrada
    (re.compile(r"\blht43\b"),         "32434"),  # modelo LHT43 (do PDF analisado)
    (re.compile(r"\b1250mm\b"),        "25913"),
    (re.compile(r"\b617mm\b"),         "25914"),
    (re.compile(r"\bpret[oa]s?\b"),    "26772"),
]

# Aceita inteiros e decimais BR com separadores de milhar: 1.234,56 | 488,77 | 123
NUMBR = r"(\d{1,3}(?:\.\d{3})*,\d{1,2}|\d+,\d{1,2}|\d+)"

# =========================
# CRIAÇÃO DA PLANILHA
# =========================
wb = Workbook()
ws = wb.active
ws["A1"] = "ORDEM"
ws["B1"] = "MATERIAL"
ws["H1"] = "QTDE"
ws["I1"] = "VALOR UNITARIO"
ws["J1"] = "TOTAL ITEM"
ws["M1"] = "DATA"
ws["O1"] = "LOJA"
ws["P1"] = "ARQUIVO"

linha = 2
log_erros = []

# =========================
# PROCESSAMENTO DOS PDFs
# =========================
processados = 0
itens_ok = 0

if not os.path.isdir(pasta_pdf):
    raise FileNotFoundError(f"Pasta não encontrada:\n{pasta_pdf}")

arquivos_pdf = [f for f in os.listdir(pasta_pdf) if f.lower().endswith(".pdf")]
if not arquivos_pdf:
    print("⚠️ Nenhum PDF encontrado na pasta informada.")

for nome_arquivo in arquivos_pdf:
    caminho_pdf = os.path.join(pasta_pdf, nome_arquivo)

    try:
        with fitz.open(caminho_pdf) as pdf:
            # Extrai texto de todas as páginas
            texto = "".join(page.get_text("text") or "" for page in pdf)

        # Extrair loja (últimos 4 do CNPJ do solicitante/destinatário)
        # Ajus.: também aceita sem o prefixo 'CNPJ:'
        cnpj_match = (
            re.search(r"CNPJ:\s*\d{2}\.\d{3}\.\d{3}/(\d{4})-\d{2}", texto) or
            re.search(r"\d{2}\.\d{3}\.\d{3}/(\d{4})-\d{2}", texto)
        )
        loja = cnpj_match.group(1) if cnpj_match else "0000"

        # Quebra o documento por itens (ex.: "It. 01 - ...")
        # Usamos split por regex para tolerar variações de espaços/dígitos
        blocos = re.split(r"\bIt\.\s*\d+\b", texto, flags=re.IGNORECASE)
        # Se não achou esse padrão, faz um split mais simples como fallback
        if len(blocos) == 1:
            blocos = texto.split("It. ")

        for bloco in blocos:
            if not bloco.strip():
                continue

            texto_bloco_norm = normaliza(bloco)

            # Detecta o material por regex dos padrões
            codigo_material = None
            for padrao, cod in material_patterns:
                if padrao.search(texto_bloco_norm):
                    codigo_material = cod
                    break

            if not codigo_material:
                # Não é um bloco do material de interesse; segue
                continue

            # === QUANTIDADE ===
            qtd = None
            # (1) padrão "n \n QUANTIDADE"
            m_before_qty = re.search(fr"{NUMBR}\s*\n\s*QUANTIDADE", bloco, re.IGNORECASE)
            if m_before_qty:
                qtd = clean_number(m_before_qty.group(1))

            # (2) padrão "QUANTIDADE \n n"
            if qtd is None:
                m_after_qty = re.search(fr"QUANTIDADE\s*\n\s*{NUMBR}", bloco, re.IGNORECASE)
                if m_after_qty:
                    qtd = clean_number(m_after_qty.group(1))

            # (3) fallback: primeira ocorrência após a palavra QUANTIDADE
            if qtd is None:
                partes = re.split(r"QUANTIDADE", bloco, flags=re.IGNORECASE)
                if len(partes) > 1:
                    numeros = re.findall(r"(\d[\d\.,]*)", partes[1])
                    if numeros:
                        qtd = clean_number(numeros[0])

            # === VALOR UNITÁRIO e TOTAL (com impostos) ===
            valor_unit = None
            total_item = None

            # Captura tripla (qtd, unitário, total) com pequenas variações de rótulos
            m_valores = re.findall(
                r"QUANTIDADE.*?(\d[\d\.,]*)[\s\S]*?"
                r"VALOR\s*UNITARIO(?:\s*COM\s*IMPOSTOS)?\s*.*?(\d[\d\.,]*)[\s\S]*?"
                r"TOTAL\s*COM\s*IMPOSTOS\s*.*?(\d[\d\.,]*)",
                bloco,
                re.IGNORECASE
            )

            if m_valores:
                grupo = m_valores[0]
                # grupo[0] também é a qtd, mas já tentamos extrair acima; usamos os dois últimos
                valor_unit = clean_number(grupo[1])
                total_item = clean_number(grupo[2])
            else:
                # fallback: depois da palavra QUANTIDADE, pega dois números em sequência
                partes = re.split(r"QUANTIDADE", bloco, flags=re.IGNORECASE)
                if len(partes) > 1:
                    nums = re.findall(r"(\d[\d\.,]*)", partes[1])
                    if len(nums) >= 2:
                        valor_unit = clean_number(nums[0])
                        total_item = clean_number(nums[1])

            # Ajuste de consistência (se houver divergência >5% e tivermos total e qtd)
            if (
                (valor_unit is None or (total_item and qtd and abs((valor_unit * qtd) - total_item) / total_item > 0.05))
                and total_item is not None and qtd is not None and qtd != 0
            ):
                valor_unit = total_item / qtd

            # Registrar linha somente se tiver quantidade e valor unitário válidos
            if qtd is not None and valor_unit is not None:
                ws[f"A{linha}"] = "F"                 # ORDEM (fixo "F" como no seu fluxo)
                ws[f"B{linha}"] = codigo_material     # MATERIAL (código mapeado)
                ws[f"H{linha}"] = qtd                 # QTDE
                ws[f"I{linha}"] = round(valor_unit, 2)# VALOR UNITARIO
                ws[f"J{linha}"] = round(total_item, 2) if total_item is not None else ""  # TOTAL ITEM
                ws[f"M{linha}"] = data_hoje           # DATA
                ws[f"O{linha}"] = loja                # LOJA (dos 4 dígitos do CNPJ)
                ws[f"P{linha}"] = nome_arquivo        # ARQUIVO
                linha += 1
                itens_ok += 1
            else:
                log_erros.append(
                    f"⚠️ Falha ao extrair (material {codigo_material}) em {nome_arquivo}\n{bloco[:400]}...\n"
                )

        processados += 1

    except Exception as e:
        log_erros.append(f"❌ Erro ao processar {nome_arquivo}: {str(e)}")

# =========================
# SALVAR PLANILHA E LOG
# =========================
caminho_planilha = os.path.join(pasta_pdf, "planilha_orcamentos_lumicenter.xlsx")
wb.save(caminho_planilha)
print(f"✅ Planilha gerada com sucesso em: {caminho_planilha}")
print(f"📄 PDFs processados: {processados} | 🧩 Itens extraídos: {itens_ok}")

if log_erros:
    caminho_log = os.path.join(pasta_pdf, "log_erros_extracao.txt")
    with open(caminho_log, "w", encoding="utf-8") as log_file:
        log_file.write("\n".join(log_erros))
    print(f"⚠️ Veja o log: {caminho_log}")


✅ Planilha gerada com sucesso em: C:\Users\126815\OneDrive - paguemenos.com.br\Área de Trabalho\ABERTURA DE REQUISIÇÃO\LOJAS NOVAS\LUMICENTER\pdfs_auto_lumi\planilha_orcamentos_lumicenter.xlsx
📄 PDFs processados: 3 | 🧩 Itens extraídos: 9


In [2]:
pip install PyMuPDF

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/19.2 MB 4.3 MB/s eta 0:00:05
   ---- ----------------------------------- 2.1/19.2 MB 5.7 MB/s eta 0:00:03
   ------- -------------------------------- 3.7/19.2 MB 6.6 MB/s eta 0:00:03
   --------- ------------------------------ 4.7/19.2 MB 7.5 MB/s eta 0:00:02
   ---------------- ----------------------- 8.1/19.2 MB 8.3 MB/s eta 0:00:02
   --------------------- ------------------ 10.5/19.2 MB 9.1 MB/s eta 0:00:01
   -------------------------- ------------- 12.6/19.2 MB 9.2 MB/s eta 0:00:01
   ----------------------------- ---------- 14.2/19.2 MB 9.0 MB/s eta 0:00:01
   --------------------------------- ------ 16.0/19.2 MB 8.9 MB/s eta 0:00:01
   ----------------------------------- ---- 17.3/19.2 MB 8.7 MB/s eta 0:00:01
   ----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
